# cop-fx-intelligence — El producto end-to-end (sistema REAL)

> Esta notebook corrió antes como *walking skeleton* (heurísticas + datos sintéticos).
> Ese esqueleto ya cumplió su función: **hoy ejecuta el sistema de producción** —
> el grafo LangGraph completo con datos y LLM reales.

**El foco del proyecto** (no ha cambiado nunca): clasificar la **dirección** del USD/COP
(`down` / `up` / `neutral`), no la magnitud, cruzando dos señales independientes —
noticias analizadas por agentes y el signo del forecast — con una capa de racionalidad
que obliga al contra-argumento y acota la confianza **por contrato Pydantic**.

| Etapa | Qué es | Dónde vive |
|---|---|---|
| 0 | Contratos Pydantic (la racionalidad vive aquí) | `src/cop_fx/contracts.py` |
| 1 | Análisis estructurado (`with_structured_output`) | `src/cop_fx/analysis/news_analyzer.py` |
| 2 | Router de materialidad (gate de costo) | `nodes.check_materiality` |
| 3 | Orchestrator-workers con `Send` (fan-out por cluster) | `nodes.orchestrate` / `topic_worker` |
| 4 | Adjudicador (reconciliación + abogado del diablo) | `nodes.adjudicate` |
| 5 | Tabla `predictions` + backtest direccional | `src/cop_fx/tracking/` |

**El producto diario NO vive en las notebooks**: vive en `cop-fx run` → `reports/report_YYYY-MM-DD.md`
y en el dashboard (`uv run streamlit run dashboard/app.py`). Las notebooks son el laboratorio.

In [1]:
# Setup — carga .env de la raíz y resuelve rutas
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
load_dotenv(ROOT / ".env")
os.chdir(ROOT)  # reports/ y data/ son relativos a la raíz

assert os.getenv("OPENAI_API_KEY", "").startswith("sk-"), "Falta OPENAI_API_KEY en .env"
print(f"Repo: {ROOT}")

Repo: /Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence


## 1. El grafo de inteligencia

Este es el sistema completo compilado. Léelo de izquierda a derecha:
dos ramas paralelas (FX y noticias), el **router** que decide si el día amerita gastar tokens,
el **fan-out dinámico** (`Send`) con un worker por cluster de tópico, la convergencia en el
**adjudicador** (`defer=True`, un solo trigger — ver el docstring de `build_graph` para el
porqué), y al final reporte → tracking → publicación.

In [2]:
from cop_fx.agents.graph import build_graph

compiled = build_graph().compile()
print(compiled.get_graph().draw_mermaid())

Importing plotly failed. Interactive plots will not work.


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	fetch_fx(fetch_fx)
	fetch_news(fetch_news)
	check_materiality(check_materiality)
	skip_news(skip_news)
	orchestrate(orchestrate)
	topic_worker(topic_worker)
	aggregate_signals(aggregate_signals)
	run_forecast(run_forecast)
	adjudicate(adjudicate<hr/><small><em>defer = True</em></small>)
	generate_report(generate_report)
	record_prediction(record_prediction)
	publish(publish)
	__end__([<p>__end__</p>]):::last
	__start__ --> fetch_fx;
	__start__ --> fetch_news;
	adjudicate --> generate_report;
	aggregate_signals --> adjudicate;
	check_materiality -.-> orchestrate;
	check_materiality -.-> skip_news;
	fetch_fx --> run_forecast;
	fetch_news --> check_materiality;
	generate_report --> record_prediction;
	orchestrate -.-> topic_worker;
	record_prediction --> publish;
	skip_news --> adjudicate;
	topic_worker --> aggregate_signals;
	publish --> __end__;
	run_forecast --> __end__;
	classDef default f

## 2. Una corrida real completa

Lo mismo que ejecuta `uv run cop-fx run` (y el cron diario de GitHub Actions).
Descarga la TRM oficial (datos.gov.co), scrapea CNN + feeds macro, y gasta
~8-10 llamadas a `gpt-5.4-mini` (≈ fracciones de centavo). Tarda ~40-60 s.

In [3]:
from cop_fx.agents.graph import run_pipeline

final_state = run_pipeline()

print("\n=== resumen de la corrida ===")
print(f"TRM actual        : {final_state.get('latest_rate', 0):,.2f} ({final_state.get('rate_change_pct', 0):+.2f}% 30d)")
print(f"Artículos crudos  : {len(final_state.get('raw_articles', []))}")
print(f"¿Día material?    : {final_state.get('has_material_news')}")
print(f"Clusters (Send)   : { {t: len(i) for t, i in final_state.get('clusters', {}).items()} }")
print(f"Errores           : {final_state.get('errors', [])}")

12:50:03  INFO      cop_fx.data.fx_fetcher  —  FX source: TRM datos.gov.co


12:50:03  INFO      cop_fx.data.fx_fetcher  —  Fetched 236 FX rows (start=2025-06-12, end=2026-06-12)


12:50:04  INFO      cop_fx.data.cnn_fetcher  —  CNNColombiaFetcher: starting (max=50)


12:50:04  WARNING   cop_fx.data.cnn_fetcher  —  RSS feed returned 0 entries: https://cnnespanol.cnn.com/colombia/feed/


12:50:04  WARNING   cop_fx.data.cnn_fetcher  —  Colombia RSS empty — trying main CNN Español feed


...

12:50:15  INFO      cop_fx.agents.nodes  —  Señal de noticias: up (score=-1.05, drivers=0)


12:50:20  INFO      cop_fx.agents.nodes  —  DirectionalCall: down (confianza=0.45, reconciliación=diverge)


12:50:20  INFO      cop_fx.tracking.predictions  —  Prediction saved: 2026-06-12 → down (0.45)


12:50:20  INFO      cop_fx.agents.nodes  —  publish: skipped (publish_enabled=False)



=== resumen de la corrida ===
TRM actual        : 3,513.54 (-2.22% 30d)
Artículos crudos  : 50
¿Día material?    : True
Clusters (Send)   : {'energy_commodities': 3, 'political_risk': 5, 'fiscal_policy': 2, 'us_global_macro': 2, 'monetary_policy': 2}
Errores           : []


## 3. El producto: `DirectionalCall`

El veredicto del adjudicador. Fíjate en tres cosas que NO son cosmética:

1. **`reconciliation`** se declara antes de decidir — si las señales divergen, la confianza
   tiene techo 0.5 *impuesto por el validador del contrato*, no por cortesía del prompt.
2. **`devils_advocate`** es obligatorio (mín. 20 caracteres): el modelo no puede emitir
   un veredicto sin construir el mejor argumento en contra.
3. **`neutral` es una salida válida**: confianza < 0.35 ⇒ el sistema se abstiene.

In [4]:
call = final_state["directional_call"]

ARROW = {"down": "⬇️ USD/COP BAJA (COP se fortalece)",
         "up": "⬆️ USD/COP SUBE (COP se debilita)",
         "neutral": "⏸️ NEUTRAL — abstención"}

print("=" * 70)
print(f"  {ARROW[call['direction']]}")
print(f"  confianza {call['confidence']:.2f} · horizonte {call['horizon_days']}d · reconciliación: {call['reconciliation']}")
print("=" * 70)
print(f"  noticias: {call['news_signal']['direction']} (score {call['news_signal']['score']})")
print(f"  serie   : {call['ts_signal']['direction']} ({call['ts_signal']['yhat_delta_pct']:+.2f}%, modelos {'concuerdan' if call['ts_signal']['models_agree'] else 'difieren'})")
print("-" * 70)
print(f"RACIONAL:\n{call['rationale']}\n")
print(f"ABOGADO DEL DIABLO:\n{call['devils_advocate']}\n")
print("CAVEATS:")
for c in call["caveats"]:
    print(f"  - {c}")

  ⬇️ USD/COP BAJA (COP se fortalece)
  confianza 0.45 · horizonte 7d · reconciliación: diverge
  noticias: up (score -1.05)
  serie   : down (-0.45%, modelos difieren)
----------------------------------------------------------------------
RACIONAL:
Las señales divergen: el news signal es alcista para USD/COP (direction up, score -1.05, aunque el score negativo sugiere presión bajista para USD/COP) mientras que el time-series signal apunta a USD/COP down con yhat_delta_pct de -0.447. Para el horizonte de 7 días, doy más peso al time-series porque es una señal direccional explícita y cuantificada sobre el corto plazo, y además la magnitud del shock informativo luce mixta y de impacto contenido: el petróleo más débil favorece al COP, pero el ruido político-electoral en Colombia y el PPI estadounidense más fuerte sostienen al dólar. En conjunto, el sesgo técnico de apreciación del COP domina levemente sobre el ruido noticioso, por lo que el veredicto final es USD/COP down.

ABOGADO DEL DIA

## 4. El reporte diario

Lo que el sistema persiste en `reports/` (y tuitearía con `--publish`).

In [5]:
from IPython.display import Markdown, display

display(Markdown(final_state["report_markdown"]))

# COP/USD Intelligence Report — 2026-06-12

## Current Rate
**1 USD = 3,513.54 COP** 📉 (-2.22% vs 30 days ago)

## Directional Call — 7 días
**⬇️ USD/COP BAJA (COP se fortalece)** · confianza **0.45** · reconciliación **diverge**

Señales: noticias = up (score -1.05) · serie = down (-0.45%, modelos difieren)

**Racional:** Las señales divergen: el news signal es alcista para USD/COP (direction up, score -1.05, aunque el score negativo sugiere presión bajista para USD/COP) mientras que el time-series signal apunta a USD/COP down con yhat_delta_pct de -0.447. Para el horizonte de 7 días, doy más peso al time-series porque es una señal direccional explícita y cuantificada sobre el corto plazo, y además la magnitud del shock informativo luce mixta y de impacto contenido: el petróleo más débil favorece al COP, pero el ruido político-electoral en Colombia y el PPI estadounidense más fuerte sostienen al dólar. En conjunto, el sesgo técnico de apreciación del COP domina levemente sobre el ruido noticioso, por lo que el veredicto final es USD/COP down.

**Abogado del diablo:** La mejor objeción es que el bloque noticioso sí contiene riesgos reales para el COP: la mayor prima política local por la segunda vuelta, el sesgo de growth más débil en Colombia y el apoyo al dólar por un PPI estadounidense más fuerte pueden pesar más que una señal técnica pequeña de -0.447%. Además, el petróleo más débil no siempre se traduce mecánicamente en COP fuerte si el mercado ya lo había descontado o si el apetito global por riesgo empeora. Bajo esa lectura, el news flow podría revertir o neutralizar el sesgo bajista del USD/COP y dejar el tipo de cambio plano o incluso al alza.

**Drivers:** (sin drivers de alta severidad)

**Caveats:**
- La señal de noticias es agregada y mezcla drivers de distinta dirección y magnitud; no hay headlines puntuales en la entrada.
- El modelo de series temporales reporta desacuerdo entre componentes (models_agree=false), así que la convicción debe ser limitada.
- La dirección del news signal y el score reportado no son perfectamente consistentes entre sí, lo que introduce ambigüedad de interpretación.
- Ventana de 7 días sensible a shocks políticos, commodities y tasas; posible alta volatilidad intraperiodo.
- Posible sesgo por artículos ya parcialmente descontados por el mercado.

## Market Narrative
[energy_commodities] La noticia más relevante para COP es la caída del petróleo, que favorece los términos de intercambio de Colombia y suele ser positiva para la moneda. El posible avance hacia una paz con Irán también mejora el apetito por riesgo, aunque su efecto sobre COP sería más indirecto y moderado. En contraste, un PPI estadounidense más fuerte alimenta expectativas de tasas altas por más tiempo, sosteniendo al dólar y limitando la apreciación del COP.
[political_risk] Las noticias concentran ruido político-electoral en Colombia, elevando la prima de riesgo por la vía de country risk. El mayor foco para USD/COP es la segunda vuelta y sus implicaciones sobre la orientación económica futura. En conjunto, el sesgo es levemente alcista para el USD/COP y bajista para el COP.
[fiscal_policy] La única señal relevante para FX es positiva: la reducción de la pobreza apunta a algo más de dinamismo económico, con canal indirecto y efecto limitado. No hay noticias de impacto cambiario material en el segundo artículo. En conjunto, el sesgo para el COP es levemente favorable pero de baja magnitud.
[us_global_macro] Las noticias apuntan a un sesgo moderadamente negativo para el crecimiento global y local, con señales de desaceleración en Colombia y Reino Unido. El canal dominante es growth, más que tasas o inflación, por lo que el impacto cambiario luce contenido. En conjunto, el USD/COP podría recibir algo de soporte por un entorno de actividad más débil y mayor cautela.
[monetary_policy] Las noticias apuntan a un choque externo por energía que eleva la inflación global y complica la política monetaria en Europa. El canal principal para USD/COP es indirecto, vía menor crecimiento global y un sesgo de mayor aversión al riesgo. En conjunto, el impacto esperado sobre el COP es moderadamente negativo y de alcance limitado.

## 7-Day Forecast
| Date | Forecast | 95% CI |
|------|----------|--------|
| 2026-06-15 | 3536.71 | 3469.64 – 3586.97 |
| 2026-06-16 | 3518.32 | 3450.53 – 3594.17 |
| 2026-06-17 | 3517.60 | 3428.82 – 3603.91 |
| 2026-06-18 | 3513.08 | 3415.54 – 3614.98 |
| 2026-06-19 | 3513.50 | 3409.12 – 3633.04 |
| 2026-06-22 | 3521.68 | 3396.48 – 3643.19 |
| 2026-06-23 | 3497.83 | 3382.49 – 3647.65 |

## Model Performance (back-test)
- **PROPHET**: MAE=59.06, RMSE=72.22, MAPE=1.65%
- **ARIMA**: MAE=8.87, RMSE=10.15, MAPE=0.25%


---
*Generated by cop-fx-intelligence at 2026-06-12T17:50:20Z*


## 5. Etapa 5 — cerrar el loop: `predictions` + backtest

La corrida de arriba ya guardó su predicción en `data/predictions.db` (nodo
`record_prediction`). Cada corrida futura **evalúa sola** las predicciones cuyo
horizonte venció, comparando contra la TRM real. Con el tiempo esto responde LA
pregunta: ¿el sistema le gana a un baseline trivial? ¿Y acierta más cuando dice
"alta confianza"? (eso valida la calibración del adjudicador).

In [6]:
from cop_fx.tracking import PredictionStore

store = PredictionStore("data/predictions.db")
preds = store.all()
print(f"{len(preds)} predicciones registradas")
preds[["run_date", "direction", "confidence", "reconciliation",
       "news_direction", "ts_direction", "latest_rate", "actual_direction", "hit"]]

1 predicciones registradas


,run_date,direction,confidence,reconciliation,news_direction,ts_direction,latest_rate,actual_direction,hit
0,2026-06-12,down,0.45,diverge,up,down,3513.54,None,None


### Backtest de la señal de serie — la prueba de honestidad (sin LLM)

Mientras la tabla `predictions` acumula historia real, podemos backtestear YA la
mitad determinista del sistema: en cada uno de los últimos N días, ¿el signo de
ARIMA predijo la dirección a 5 días? ¿Le gana a `momentum` (repetir el último
movimiento) y a `always_up`? **Si no les gana, la serie no aporta y todo el peso
del sistema recae en las noticias** — eso es exactamente lo que queremos saber.

In [7]:
import pandas as pd

from cop_fx.tracking import directional_backtest

detail, summary = directional_backtest(final_state["fx_df"], horizon_days=5, n_origins=40)

print(f"Backtest: {summary['n_origins']} orígenes, horizonte {summary['horizon_days']}d\n")
rows = []
for strategy in ("arima", "momentum", "always_up"):
    s = summary[strategy]
    rows.append({
        "estrategia": strategy,
        "hit_rate": f"{s['hit_rate']:.0%}" if s["hit_rate"] is not None else "—",
        "decisiones": s["n_decided"],
        "abstenciones": s["n_abstained"],
    })
pd.DataFrame(rows)

12:50:20  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3664.41, day+5=3670.42


12:50:20  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3678.19, day+5=3682.31


12:50:20  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3640.63, day+5=3640.30


12:50:20  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3642.93, day+5=3636.56


/Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/carlosdaniel/Documents/Projects/Personales/AI_APPS/cop-fx-intelligence/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


...

12:50:22  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3560.24, day+5=3559.13


12:50:22  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3562.00, day+5=3572.94


12:50:22  INFO      cop_fx.timeseries.models  —  ARIMA forecast: last known=3572.85, day+5=3567.83


Backtest: 40 orígenes, horizonte 5d



,estrategia,hit_rate,decisiones,abstenciones
0,arima,52%,25,15
1,momentum,53%,36,4
2,always_up,42%,40,0


In [8]:
# ¿Cómo se distribuyen los aciertos en el tiempo?
detail["arima_hit"] = (detail["arima"] == detail["actual"]) & (detail["arima"] != "neutral")
print(detail.tail(10).to_string(index=False))

    origin actual   arima momentum always_up  arima_hit
2026-05-21   down neutral     down        up      False
2026-05-22   down neutral     down        up      False
2026-05-23   down      up     down        up      False
2026-05-27   down neutral     down        up      False
2026-05-28   down      up     down        up      False
2026-05-29   down neutral       up        up      False
2026-05-30   down    down       up        up       True
2026-06-02     up neutral     down        up      False
2026-06-03     up      up  neutral        up       True
2026-06-04   down    down       up        up       True


---
## Qué sigue

- **Etapa 6**: checkpointer (`SqliteSaver`), `interrupt` antes de publicar (HITL: tú apruebas
  el tweet) y memoria entre corridas ("ayer dije down con 0.7 y fallé" como contexto del
  adjudicador). Curso: módulos 06/07/12.
- **GCP** (fases 5-6 de `docs/arquitectura.md`): Cloud Run Jobs + Scheduler + BigQuery —
  swaps de adapter, nada de lo de arriba se reescribe.

**Para replicar y dominar LangGraph**: reconstruye este grafo desde cero en una notebook
vacía, en este orden — (1) un nodo con `with_structured_output`, (2) el router con
`add_conditional_edges`, (3) el fan-out con `Send` y reducers `operator.add`,
(4) el join con `defer=True` y UN solo trigger. Cada paso tiene su test en
`tests/unit/test_graph_nodes.py` para verificarte.